<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Ground Projection

Detected line segments are transformed from the camera’s image plane to the ground plane using the homography matrix obtained during the [Duckiebot's extrinsic camera calibration process](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-camera-calibration.html). This is achieved by the [ground_projection package](../packages/dt-ros2-core/packages/ground_projection/) (node: [`ground_projection/ground_projection_node.py`](../packages/dt-ros2-core/packages/ground_projection/ground_projection/ground_projection_node.py)). If you have not performed the camera calibration for your physical Duckiebot, you should do this before moving on.

Virtual Duckiebots have a default calibration that will work out of the box, but nonetheless you can enjoy practicing the calibration procedure by loading the `sandbox` map in the Duckiematrix and entering the garage.

## Homography matrix and camera calibration

The homography matrix, which establishes the spatial relationship between the camera and the ground plane, is obtained through an [extrinsic calibration procedure](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-extrinsics-camera-calibration.html).

The transformation is represented as $P_w = H P_c$, where $P_w = (x, y, 1)$ and $P_c = (u, v, 1)$. $(u, v)$ are the pixel coordinates in the image, $(x, y)$ are the corresponding coordinates in the ground plane, and $H$ is the homography matrix.

Additional details on planar homographies, and how to estimate them, are available in the [Duckietown Projective Geometry](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/vision/06-full-lecture-vision.html#projective-geometry) lecture.

## Input to the ground projection node and rectification

The input to the ground projection step is the list of detected segments from the line detector (the node subscribes to `ground_projection_node/lineseglist_in`, which the demo launch file remaps to `line_detector_node/segment_list`, and to the camera intrinsics on `camera_node/camera_info`). As the careful observer might have noticed, the Duckiebot's camera is equipped with a fisheye lens, which offers extended field of view at the cost of non-trivial image distortion.

Line segments are therefore first rectified using the intrinsic parameters (from the [intrinsic camera calibration procedure](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-intrinsics-camera-calibration.html)) to correct for distortions, ensuring accurate projection in the 3D world.

## Output

Since the identified segments are edges of the lane markings, if Duckietown are built according to the [Duckietown appearance specifications](https://docs.duckietown.com/ente/duckietown-manual/30-duckietown-city/appearance-specifications/duckietown-smart-city-appearance-specifications-introduction.html), correctly projected line segments will be represented in the ground plane as parallel lines. The projected segments are published on `ground_projection_node/lineseglist_out`, which is what the lane filter consumes.

The reprojection image proves a useful representation for downstream tasks such as lane localization and control, as it shows how the robots is perceiving the (road) environment around it.

Additional images showing rectified and projected views are published for debugging and verification (`ground_projection_node/debug/projected_image/compressed` and `ground_projection_node/debug/projected_image/rectified/compressed`).

## Verifying the ground projections

To view the reprojection of the line segment markers on the road, look at the `ground_projection_node/debug/ground_projection_image/compressed` topic. As in the previous notebook, you can do it right here with an `ImageView` panel (the demo must be running on your robot), or with `dts duckiebot image_viewer ROBOT_NAME`:


In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ImageView

ROBOT_NAME = "ROBOT_NAME"   # <-- the name of your (real or virtual) Duckiebot
ros = RosSession(ROBOT_NAME)  # connects to the robot's zenoh router and spins in the background

ImageView(ros, "ground_projection_node/debug/ground_projection_image/compressed")


You should see a top-down view of the lane in front of your Duckiebot. The closer to the Duckiebot, the more parallel the lines should look. Distant segments always carry high uncertainty, especially due to the low resolution of the input image.

You can use the ground projection image to verify visually that the homography calculated during the extrinsic calibration is correct. High convergent or divergent lines would suggest the need to re-perform the camera calibration procedure.

![projection](../assets/images/projection.png)

There are no parameters to tune at this stage as they were all automatically during the camera calibration procedures (the only configuration of the node, [`config/ground_projection_node/default.yaml`](../packages/dt-ros2-core/packages/ground_projection/config/ground_projection_node/default.yaml), describes the region of the ground drawn in the debug image).

For much more details about how the homography works, refer to the second notebook of the [computer vision learning experience](https://github.com/duckietown/lx-computer-vision/), and/or the [Duckietown Projetive Geometry](https://docs.duckietown.com/ente/duckietown-manual/80-instructor-manual/available-resources/slides/vision/06-full-lecture-vision.html#projective-geometry) lecture.

The next step in the pipeline is for the robot to leverage the estimated position of the lane in the world to estimate its pose (position and orientation) with respect to it. We explore this in the next notebook: [lane localization](./03_lane_localization.ipynb).
